In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
# Autosomal genetic length (cM) of the map used in the simulations.
# hap-ibd doesn't report segment cM positions, so this is the 22-autosome span
# measured from the IBIS SNP panel segment results on the same data.
GENOME_CM = 3272.9

fo_rels = ['Parent Child', 'Full sibling', 'Half Sibling']
anc_rels = ['Grandparent', 'Great Grandparent', 'Great Great Grandparent', 'Great Great Great Grandparent']
full_cuz_rels = ['Full First Cousin', 'Full Second Cousin', 'Full Third Cousin', 'Full Fourth Cousin', 'Full Fifth Cousin']
half_cuz_rels = ['Half First Cousin', 'Half Second Cousin', 'Half Third Cousin', 'Half Forth Cousin', 'Half Fifth Cousin']

expected_kinship = {
    'Parent Child': 1/4, 'Full sibling': 1/4, 'Half Sibling': 1/8,
    'Grandparent': 1/8, 'Great Grandparent': 1/16, 'Great Great Grandparent': 1/32, 'Great Great Great Grandparent': 1/64,
    'Full First Cousin': 1/16, 'Full Second Cousin': 1/64, 'Full Third Cousin': 1/256, 'Full Fourth Cousin': 1/1024, 'Full Fifth Cousin': 1/4096,
    'Half First Cousin': 1/32, 'Half Second Cousin': 1/128, 'Half Third Cousin': 1/512, 'Half Forth Cousin': 1/2048, 'Half Fifth Cousin': 1/8192,
}
# Summed over haplotype pairs, total cM = 4 x kinship x genome length
# (= 2 x PropIBD x genome length), e.g. Parent Child = 1 x genome, Half Sibling = 0.5 x genome
expected_total_cM = {rel: 4 * k * GENOME_CM for rel, k in expected_kinship.items()}


def plot_rels(df, y, rels, expected, title, fname):
    """Violin plot of `y` by relationship, colored by superpopulation,
    with a red dashed expected line spanning each relationship's own category."""
    sub = df[df['RC'].isin(rels)]
    fig = px.violin(sub, y=y, x='RC', color='pop', hover_data=sub.columns,
                    template='simple_white', category_orders={'RC': rels})
    for i, rel in enumerate(rels):
        fig.add_shape(type='line', xref='x', x0=i-0.5, x1=i+0.5, yref='y',
                      y0=expected[rel], y1=expected[rel],
                      line=dict(color='red', dash='dash', width=2))
    fig.update_layout(xaxis_title=' ', yaxis_title='Total shared cM', title=title)
    fig.write_image(f'images/{fname}.png', scale=2)
    fig.show()

# hap-ibd Segments (Total Shared cM)

Segment lengths are summed over all haplotype pairs for each pair of individuals. On this scale the expectation is 4 × kinship × genome length, so Parent Child and Full sibling ≈ one genome, and Half Sibling and Grandparent ≈ half. Pairs with no detected segments aren't in the file, so distant relationships only include pairs where hap-ibd found something.

In [ ]:
# One row per haplotype segment, so collapse to one row per pair.
# Read in chunks with only the needed columns to keep memory down (multi-GB file).
keys = ['pop', 'seed', 'Fam_ID', 'ID1', 'ID2']
usecols = keys + ['RC', 'CM_Length']

parts = []
for chunk in pd.read_csv('hap_ibd_snp_panel_results.csv', usecols=usecols, chunksize=2_000_000):
    # order-invariant pair, in case (1, 4) ever shows up as (4, 1)
    a, b = chunk['ID1'].to_numpy(), chunk['ID2'].to_numpy()
    chunk['ID1'], chunk['ID2'] = np.minimum(a, b), np.maximum(a, b)
    parts.append(chunk.groupby(keys + ['RC'], sort=False)
                      .agg(n_seg=('CM_Length', 'size'),
                           total_hap_cM=('CM_Length', 'sum'),
                           max_seg_cM=('CM_Length', 'max')))

# a pair can straddle two chunks, so combine again
hap_ibd_pairs = (pd.concat(parts)
                   .groupby(level=list(range(len(keys) + 1)))
                   .agg({'n_seg': 'sum', 'total_hap_cM': 'sum', 'max_seg_cM': 'max'})
                   .reset_index())
hap_ibd_pairs['RC'] = hap_ibd_pairs['RC'].str.strip()

# PropIBD-scale equivalent, for comparing against KING / IBIS in the hover data
hap_ibd_pairs['PropIBD_equiv'] = hap_ibd_pairs['total_hap_cM'] / (2 * GENOME_CM)
hap_ibd_pairs

In [ ]:
hap_ibd_pairs['RC'].value_counts()

## First Order Relationships

In [ ]:
plot_rels(hap_ibd_pairs, 'total_hap_cM', fo_rels, expected_total_cM,
          'hap-ibd Total cM, First Order, SNP Panel', 'hap_ibd_first_order_snp_panel')

## Direct Ancestors

In [ ]:
plot_rels(hap_ibd_pairs, 'total_hap_cM', anc_rels, expected_total_cM,
          'hap-ibd Total cM, Direct Ancestors, SNP Panel', 'hap_ibd_direct_ancestors_snp_panel')

## Full Cousins

In [ ]:
plot_rels(hap_ibd_pairs, 'total_hap_cM', full_cuz_rels, expected_total_cM,
          'hap-ibd Total cM, Full Cousins, SNP Panel', 'hap_ibd_cousin_snp_panel')

## Half Cousins

In [ ]:
plot_rels(hap_ibd_pairs, 'total_hap_cM', half_cuz_rels, expected_total_cM,
          'hap-ibd Total cM, Half Cousins, SNP Panel', 'hap_ibd_half_rels_snp_panel')